# Clustering — Interactive Notebook
## TTK4260: Multivariate Data Analysis & Machine Learning
**Prof. Adil Rasheed** — Department of Engineering Cybernetics, NTNU  
**Spring 2026**

---

This notebook provides an **interactive, hands-on companion** to the Clustering lecture. We will cover:

1. **Motivation and Foundations** — distances, scaling, curse of dimensionality  
2. **K-Means** — algorithm, initialisation, elbow method, silhouette analysis  
3. **Hierarchical Clustering** — agglomerative algorithm, linkage criteria, dendrograms  
4. **DBSCAN** — density-based clustering, core/border/noise points, $k$-distance plot  
5. **Gaussian Mixture Models (GMMs)** — soft assignments, EM algorithm, BIC  
6. **Method Comparison** — side-by-side on challenging datasets  
7. **Evaluation and Pitfalls** — internal/external metrics, common mistakes

Each section includes **visualisations with plot-specific explanations** that reference the generated data.

## 0. Setup and Imports

We begin by importing all the libraries we need. The notebook uses `scikit-learn` for clustering algorithms, `matplotlib` and `seaborn` for plotting, and `scipy` for hierarchical clustering utilities.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
import warnings
warnings.filterwarnings('ignore')

# Consistent style
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
    'legend.fontsize': 10,
    'figure.dpi': 100,
})

# Reproducibility
np.random.seed(42)

# NTNU-inspired colour palette
NTNU_BLUE = '#00509e'
NTNU_COLORS = ['#00509e', '#e63946', '#2a9d8f', '#e9c46a', '#7b2d8e', '#f4a261']

print("All imports successful — ready to cluster!")

---
# 1. Motivation and Foundations

## What is Clustering?

Clustering is the task of **grouping similar observations when no labels are given** — a central problem in unsupervised learning. The goal is to discover structure, summarise data, or generate hypotheses.

There is **no single universal definition** of a cluster. Different algorithms encode different assumptions:
- A **compact** group of points (K-Means)
- A **well-separated** set (hierarchical methods)
- A **dense region** surrounded by sparse regions (DBSCAN)
- A component of a **probability mixture** (GMMs)

Let us start by creating a simple synthetic dataset to visualise what clusters look like.

In [ ]:
# Generate a simple 3-cluster dataset
X_simple, y_simple = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: raw unlabelled data (as we receive it)
axes[0].scatter(X_simple[:, 0], X_simple[:, 1], c='gray', alpha=0.6, edgecolors='k', linewidth=0.3, s=40)
axes[0].set_title('Raw Data (No Labels)')
axes[0].set_xlabel('$x_1$'); axes[0].set_ylabel('$x_2$')

# Right: ground-truth colouring
scatter = axes[1].scatter(X_simple[:, 0], X_simple[:, 1], c=y_simple,
                          cmap=ListedColormap(NTNU_COLORS[:3]), alpha=0.7, edgecolors='k', linewidth=0.3, s=40)
axes[1].set_title('Ground Truth (3 Clusters)')
axes[1].set_xlabel('$x_1$'); axes[1].set_ylabel('$x_2$')
axes[1].legend(*scatter.legend_elements(), title='Cluster')

plt.tight_layout()
plt.show()

**What the plot tells us:** The left panel shows the data as an algorithm would see it — no colours, no labels. Even to the human eye, three distinct groupings are visible. The right panel reveals the ground truth. The three clusters are well-separated and roughly spherical, making this an "easy" dataset for most algorithms. We will later test on harder datasets where clusters overlap, have non-convex shapes, or vary in density.

## 1.1 Similarity and Distance

Clustering results depend heavily on how we measure **similarity** between points. Let $\boldsymbol{x}, \boldsymbol{y} \in \mathbb{R}^{n}$ be two feature vectors. Common metrics include:

- **Euclidean distance:** $d(\boldsymbol{x}, \boldsymbol{y}) = \|\boldsymbol{x} - \boldsymbol{y}\|_2 = \sqrt{\sum_{j=1}^{n}(x_j - y_j)^2}$
- **Manhattan distance:** $d(\boldsymbol{x}, \boldsymbol{y}) = \|\boldsymbol{x} - \boldsymbol{y}\|_1 = \sum_{j=1}^{n}|x_j - y_j|$
- **Cosine similarity:** $\cos\theta = \frac{\boldsymbol{x}^\top \boldsymbol{y}}{\|\boldsymbol{x}\|\;\|\boldsymbol{y}\|}$

Let us visualise how Euclidean and Manhattan distances differ between two points.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

x_pt = np.array([1.5, 4.0])
y_pt = np.array([5.0, 1.5])

# Points
ax.plot(*x_pt, 'o', color=NTNU_BLUE, markersize=12, zorder=5)
ax.annotate(r'$\mathbf{x}$', x_pt, textcoords="offset points", xytext=(-15, 10),
            fontsize=14, fontweight='bold', color=NTNU_BLUE)
ax.plot(*y_pt, 'o', color='#e63946', markersize=12, zorder=5)
ax.annotate(r'$\mathbf{y}$', y_pt, textcoords="offset points", xytext=(10, -5),
            fontsize=14, fontweight='bold', color='#e63946')

# Euclidean (straight line)
eucl = np.linalg.norm(x_pt - y_pt)
ax.plot([x_pt[0], y_pt[0]], [x_pt[1], y_pt[1]], '--', color='gray', linewidth=2, label=f'Euclidean = {eucl:.2f}')

# Manhattan (L-shaped path)
manh = np.abs(x_pt - y_pt).sum()
ax.plot([x_pt[0], y_pt[0], y_pt[0]], [x_pt[1], x_pt[1], y_pt[1]],
        ':', color='#e9c46a', linewidth=3, label=f'Manhattan = {manh:.2f}')

# Cosine angle
cos_sim = np.dot(x_pt, y_pt) / (np.linalg.norm(x_pt) * np.linalg.norm(y_pt))
angle = np.degrees(np.arccos(np.clip(cos_sim, -1, 1)))

# Draw angle arc from origin
theta_x = np.degrees(np.arctan2(x_pt[1], x_pt[0]))
theta_y = np.degrees(np.arctan2(y_pt[1], y_pt[0]))
arc_angles = np.linspace(np.radians(theta_y), np.radians(theta_x), 30)
arc_r = 1.2
ax.plot(arc_r * np.cos(arc_angles), arc_r * np.sin(arc_angles), '-', color='#2a9d8f', linewidth=2)
ax.annotate(f'θ = {angle:.1f}°\ncos θ = {cos_sim:.3f}', xy=(1.0, 1.0),
            fontsize=10, color='#2a9d8f')

# Vectors from origin
ax.annotate('', xy=x_pt, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color=NTNU_BLUE, lw=1.5))
ax.annotate('', xy=y_pt, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='#e63946', lw=1.5))

ax.set_xlim(-0.5, 6); ax.set_ylim(-0.5, 5.5)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Distance Metrics: Euclidean vs. Manhattan vs. Cosine')
ax.legend(loc='upper right', fontsize=11)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**What the plot tells us:** The Euclidean distance (dashed gray line) is the straight-line "as the crow flies" distance between $\boldsymbol{x}$ and $\boldsymbol{y}$, measuring about 4.27 units. The Manhattan distance (dotted yellow L-shaped path) is longer at 6.00 — it measures the distance along axes only, like walking on a city grid. The cosine similarity (green arc) measures the *angle* between the vectors from the origin; at ~0.69 the vectors point in moderately similar directions. The key takeaway: for clustering, **the choice of distance changes which points are considered "nearby"**, and thus which clusters emerge.

## 1.2 Why Scaling Matters

If features have vastly different scales, features with larger numeric ranges will **dominate** the distance calculation. Consider clustering people by annual income (range ~30,000–200,000) and age (range ~20–70). Without standardisation, income overwhelms age.

**Best practice:** Standardise each feature: $z_j = \frac{x_j - \mu_j}{\sigma_j}$

In [ ]:
np.random.seed(42)

# Simulate: 2 groups differing mainly in age, but income has much larger range
n_per = 60
# Group 1: younger, mixed income
age1 = np.random.normal(28, 4, n_per)
income1 = np.random.normal(80000, 35000, n_per)
# Group 2: older, mixed income
age2 = np.random.normal(52, 5, n_per)
income2 = np.random.normal(110000, 40000, n_per)

X_raw = np.vstack([np.column_stack([age1, income1]),
                    np.column_stack([age2, income2])])
y_true_scale = np.array([0]*n_per + [1]*n_per)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (data, title) in enumerate([
    (X_raw, 'Before Scaling'),
    (StandardScaler().fit_transform(X_raw), 'After Standardisation')
]):
    km = KMeans(n_clusters=2, random_state=42, n_init=10).fit(data)
    labels = km.labels_
    ari = adjusted_rand_score(y_true_scale, labels)
    
    ax = axes[ax_idx]
    for k in range(2):
        mask = labels == k
        ax.scatter(data[mask, 0], data[mask, 1], c=NTNU_COLORS[k], alpha=0.6,
                   edgecolors='k', linewidth=0.3, s=40, label=f'Cluster {k}')
        ax.scatter(km.cluster_centers_[k, 0], km.cluster_centers_[k, 1],
                   marker='X', s=200, c=NTNU_COLORS[k], edgecolors='black', linewidth=1.5, zorder=5)
    ax.set_title(f'{title}\nK-Means ARI = {ari:.2f}')
    if ax_idx == 0:
        ax.set_xlabel('Age'); ax.set_ylabel('Income')
    else:
        ax.set_xlabel('$z_{\\mathrm{age}}$'); ax.set_ylabel('$z_{\\mathrm{income}}$')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**What the plot tells us:** In the left panel (unscaled), K-Means splits the data horizontally by income level — it completely ignores age because income values are ~1000× larger in magnitude. The ARI is low, meaning the clusters do not match the true age-based groups at all. In the right panel (standardised), both features contribute equally to the distance. K-Means now correctly separates the younger from the older group, producing a much higher ARI. This dramatically illustrates why **standardisation is essential** for distance-based clustering.

## 1.3 The Curse of Dimensionality

In high dimensions, distances become **less discriminative**: the ratio $d_{\max}/d_{\min} \to 1$ as dimensionality grows. This means all points appear roughly equidistant, making clustering unreliable.

In [ ]:
dims = np.arange(2, 502, 10)
ratios = []
n_pts = 200

for d in dims:
    pts = np.random.randn(n_pts, d)
    dists = np.sqrt(((pts[:, None, :] - pts[None, :, :]) ** 2).sum(axis=-1))
    # Exclude self-distances
    mask = ~np.eye(n_pts, dtype=bool)
    d_vals = dists[mask]
    ratios.append(d_vals.max() / d_vals.min())

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(dims, ratios, '-o', color=NTNU_BLUE, markersize=3, linewidth=2)
ax.axhline(y=1.0, color='#e63946', linestyle='--', linewidth=1.5, label='$d_{\\max}/d_{\\min} = 1$ (all equidistant)')
ax.set_xlabel('Dimensionality $n$')
ax.set_ylabel('$d_{\\max} \\,/\\, d_{\\min}$')
ax.set_title('Curse of Dimensionality: Distance Ratio Collapses')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**What the plot tells us:** At low dimensions ($n \approx 2$–$20$), the max-to-min distance ratio is high, meaning some points are genuinely much closer than others — clustering can distinguish them. As dimensionality grows, this ratio drops steeply and approaches 1.0 (red dashed line). By $n \approx 200$+, the farthest point is barely farther than the nearest one. **This is why dimensionality reduction (e.g. PCA, feature selection) is often critical before clustering high-dimensional data.**

---
# 2. K-Means Clustering

## 2.1 The Basic Idea

K-Means represents each cluster by its **centroid** $\boldsymbol{\mu}_k$ and assigns every point to its nearest centroid. It minimises the **within-cluster sum of squares** (WCSS):

$$J = \sum_{k=1}^{K} \sum_{\boldsymbol{x}_i \in C_k} \|\boldsymbol{x}_i - \boldsymbol{\mu}_k\|^2$$

### The Algorithm
1. **Choose** the number of clusters $K$
2. **Initialise** $K$ centroids $\boldsymbol{\mu}_1, \ldots, \boldsymbol{\mu}_K$
3. **Assign** each point to the nearest centroid: $c_i = \arg\min_k \|\boldsymbol{x}_i - \boldsymbol{\mu}_k\|^2$
4. **Update** centroids: $\boldsymbol{\mu}_k \leftarrow \frac{1}{|C_k|} \sum_{\boldsymbol{x}_i \in C_k} \boldsymbol{x}_i$
5. **Repeat** steps 3–4 until convergence

Let us animate this iterative process.

In [ ]:
# Step-by-step K-Means visualisation
X_km, y_km = make_blobs(n_samples=200, centers=4, cluster_std=1.0, random_state=10)

# Manual K-Means to capture iterations
K = 4
rng = np.random.RandomState(3)
centroids = X_km[rng.choice(len(X_km), K, replace=False)]
history = [centroids.copy()]

for iteration in range(8):
    # Assign
    dists = np.linalg.norm(X_km[:, None] - centroids[None, :], axis=2)
    labels = dists.argmin(axis=1)
    # Update
    new_centroids = np.array([X_km[labels == k].mean(axis=0) if (labels == k).any() else centroids[k]
                               for k in range(K)])
    if np.allclose(new_centroids, centroids):
        history.append(new_centroids.copy())
        break
    centroids = new_centroids
    history.append(centroids.copy())

# Plot selected iterations
iters_to_show = [0, 1, 2, len(history)-1]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, it_idx in zip(axes, iters_to_show):
    c = history[it_idx]
    dists = np.linalg.norm(X_km[:, None] - c[None, :], axis=2)
    labs = dists.argmin(axis=1)
    
    for k in range(K):
        mask = labs == k
        ax.scatter(X_km[mask, 0], X_km[mask, 1], c=NTNU_COLORS[k], alpha=0.5, s=30, edgecolors='k', linewidth=0.2)
    
    # Centroids
    for k in range(K):
        ax.scatter(c[k, 0], c[k, 1], marker='X', s=200, c=NTNU_COLORS[k],
                   edgecolors='black', linewidth=2, zorder=5)
    
    # Draw trails from previous centroid
    if it_idx > 0:
        prev = history[it_idx - 1] if it_idx < len(history) else history[-2]
        for k in range(K):
            ax.annotate('', xy=c[k], xytext=prev[k],
                        arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    
    label = f'Iteration {it_idx}' if it_idx < len(history)-1 else f'Converged (iter {it_idx})'
    ax.set_title(label)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('K-Means Algorithm: Step-by-Step Convergence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**What the plot tells us:** At Iteration 0, the centroids (large ×-markers) are randomly placed and the resulting assignments are poor — some clusters have stolen points from others. By Iteration 1, the centroids have already moved significantly toward the centres of their assigned groups. At Iteration 2, the assignments are nearly correct. By convergence, each centroid sits at the mean of its cluster and no point changes assignment. This illustrates K-Means' alternating assign/update strategy, which typically converges within 5–15 iterations on well-structured data like this.

## 2.2 Initialisation Matters: K-Means++

Different initial centroids can lead to **different local minima**. K-Means++ addresses this by selecting initial centroids that are **spread out**:
1. Pick $\boldsymbol{\mu}_1$ uniformly at random from the data
2. Pick $\boldsymbol{\mu}_{k+1}$ with probability proportional to $d(\boldsymbol{x}_i, \text{nearest existing centroid})^2$
3. Repeat until $K$ centroids are chosen

Let us compare random vs. K-Means++ initialisation across multiple runs.

In [ ]:
X_init, _ = make_blobs(n_samples=300, centers=5, cluster_std=1.2, random_state=7)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
n_runs = 20

for ax, init_method, title in zip(axes, ['random', 'k-means++'],
                                    ['Random Initialisation', 'K-Means++ Initialisation']):
    wcss_vals = []
    for run in range(n_runs):
        km = KMeans(n_clusters=5, init=init_method, n_init=1, random_state=run).fit(X_init)
        wcss_vals.append(km.inertia_)
    
    color = '#e63946' if init_method == 'random' else NTNU_BLUE
    ax.bar(range(n_runs), wcss_vals, color=color, alpha=0.7, edgecolor='k', linewidth=0.3)
    ax.axhline(min(wcss_vals), color='#2a9d8f', linestyle='--', linewidth=2, label=f'Best WCSS = {min(wcss_vals):.0f}')
    ax.set_xlabel('Run')
    ax.set_ylabel('WCSS (Inertia)')
    ax.set_title(f'{title}\nStd dev = {np.std(wcss_vals):.1f}')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

**What the plot tells us:** With random initialisation (left), the WCSS varies considerably across the 20 runs — some runs get stuck in poor local minima with high inertia. The standard deviation is large. With K-Means++ (right), almost every run converges to nearly the same (optimal) WCSS value. The bars are nearly uniform in height and the standard deviation is much lower. This confirms that **K-Means++ provides much more consistent and reliable results**, which is why `scikit-learn` uses it as the default.

## 2.3 Choosing $K$: The Elbow Method

WCSS always decreases as $K$ grows (trivially, $J=0$ when $K=m$). The **elbow method** looks for the $K$ where the rate of decrease sharply slows down.

In [ ]:
X_elbow, y_elbow = make_blobs(n_samples=400, centers=4, cluster_std=0.9, random_state=42)

K_range = range(1, 11)
wcss = []
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_elbow)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(K_range, wcss, 'o-', color=NTNU_BLUE, linewidth=2, markersize=8)

# Highlight the elbow at K=4
ax.plot(4, wcss[3], 'o', color='#e63946', markersize=15, zorder=5)
ax.annotate('Elbow at $K=4$', xy=(4, wcss[3]), xytext=(5.5, wcss[2]),
            fontsize=12, color='#e63946', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#e63946', lw=2))

ax.set_xlabel('Number of Clusters $K$')
ax.set_ylabel('WCSS (Inertia)')
ax.set_title('Elbow Method for Choosing $K$')
ax.set_xticks(range(1, 11))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**What the plot tells us:** The WCSS drops dramatically from $K=1$ to $K=4$, then the improvement becomes marginal for $K \geq 5$. This creates a clear "elbow" at $K=4$, which matches the 4 clusters we generated. Going from $K=4$ to $K=5$ barely reduces WCSS — the additional cluster is not capturing any new structure, just subdividing an existing group. Note: the elbow is not always this clear; on more complex data, it can be ambiguous.

## 2.4 Choosing $K$: Silhouette Analysis

For each point $\boldsymbol{x}_i$, the **silhouette score** measures how well it fits its assigned cluster:

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}} \in [-1, 1]$$

- $a(i)$ = mean distance to points in the same cluster
- $b(i)$ = smallest mean distance to any other cluster
- $s(i) \approx +1$: well clustered, $s(i) \approx 0$: on boundary, $s(i) < 0$: likely misclassified

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, k in enumerate([2, 3, 4, 5, 6, 7]):
    ax = axes[idx // 3, idx % 3]
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_elbow)
    labels = km.labels_
    sil_avg = silhouette_score(X_elbow, labels)
    sample_sils = silhouette_samples(X_elbow, labels)
    
    y_lower = 10
    for i in range(k):
        cluster_sils = np.sort(sample_sils[labels == i])
        size_cluster_i = cluster_sils.shape[0]
        y_upper = y_lower + size_cluster_i
        
        color = plt.cm.Set2(i / k)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sils,
                          facecolor=color, edgecolor=color, alpha=0.7)
        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i), fontsize=9, fontweight='bold')
        y_lower = y_upper + 10
    
    ax.axvline(x=sil_avg, color='#e63946', linestyle='--', linewidth=2)
    ax.set_title(f'$K = {k}$ | Avg Silhouette = {sil_avg:.3f}')
    ax.set_xlabel('Silhouette coefficient')
    ax.set_ylabel('Cluster label')
    ax.set_xlim([-0.15, 1.0])
    ax.set_yticks([])

plt.suptitle('Silhouette Analysis for Different $K$ Values', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**What the plot tells us:** Each horizontal "knife shape" represents one cluster; the width indicates individual silhouette scores and the red dashed line is the overall average. At $K=4$, the average silhouette is highest and all four clusters have relatively uniform, wide silhouette profiles — indicating well-defined, balanced clusters. At $K=2$ or $K=3$, some clusters are too broad (forcing dissimilar points together). At $K=5$, $6$, $7$, we see thin clusters and more points with low or negative silhouette values — the algorithm is over-splitting. **$K=4$ maximises the silhouette score**, confirming the elbow method's suggestion.

## 2.5 Limitations of K-Means

K-Means assumes **convex, roughly spherical** clusters of similar size. It fails on non-convex shapes like half-moons or concentric circles.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Three challenging datasets
datasets = {
    'Half-Moons': make_moons(n_samples=400, noise=0.06, random_state=42),
    'Concentric Circles': make_circles(n_samples=400, noise=0.05, factor=0.5, random_state=42),
    'Anisotropic Blobs': (None, None),  # created below
}

# Anisotropic blobs
X_aniso, y_aniso = make_blobs(n_samples=400, centers=3, cluster_std=0.5, random_state=42)
transformation = [[0.6, -0.6], [-0.4, 0.8]]
X_aniso = np.dot(X_aniso, transformation)
datasets['Anisotropic Blobs'] = (X_aniso, y_aniso)

for col, (name, (X_d, y_d)) in enumerate(datasets.items()):
    n_true = len(np.unique(y_d))
    km = KMeans(n_clusters=n_true, n_init=10, random_state=42).fit(X_d)
    
    # Top row: true labels
    axes[0, col].scatter(X_d[:, 0], X_d[:, 1], c=y_d, cmap=ListedColormap(NTNU_COLORS[:n_true]),
                          alpha=0.6, s=25, edgecolors='k', linewidth=0.2)
    axes[0, col].set_title(f'{name} — True Labels')
    axes[0, col].set_xticks([]); axes[0, col].set_yticks([])
    
    # Bottom row: K-Means result
    ari = adjusted_rand_score(y_d, km.labels_)
    axes[1, col].scatter(X_d[:, 0], X_d[:, 1], c=km.labels_, cmap=ListedColormap(NTNU_COLORS[:n_true]),
                          alpha=0.6, s=25, edgecolors='k', linewidth=0.2)
    axes[1, col].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                          marker='X', s=150, c='black', zorder=5)
    axes[1, col].set_title(f'K-Means (ARI = {ari:.2f})')
    axes[1, col].set_xticks([]); axes[1, col].set_yticks([])

axes[0, 0].set_ylabel('Ground Truth', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('K-Means Result', fontsize=12, fontweight='bold')
plt.suptitle('K-Means Limitations: Non-Convex and Anisotropic Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**What the plot tells us:** K-Means completely fails on the half-moons (ARI ≈ 0.00) — it draws a vertical line between them rather than following the crescent shapes. On the concentric circles, it again cannot separate the inner ring from the outer one since they are not linearly separable. On anisotropic (elongated) blobs, the performance is better but still imperfect — the centroids are pulled toward the centres of mass, ignoring the elongated shape. These failures all stem from K-Means' assumption that clusters are **compact and spherical**.

---
# 3. Hierarchical Clustering

## 3.1 Agglomerative Clustering and Dendrograms

Hierarchical clustering builds a **tree of nested clusterings** (a **dendrogram**) by iteratively merging the two closest clusters.

The key question is: how do we define "distance between clusters"? This is the **linkage criterion**:
- **Single linkage** (nearest): $d(A,B) = \min_{x_i \in A, x_j \in B} \|x_i - x_j\|$
- **Complete linkage** (farthest): $d(A,B) = \max_{x_i \in A, x_j \in B} \|x_i - x_j\|$
- **Average linkage**: mean of all pairwise distances
- **Ward's linkage**: merge that minimally increases WCSS (usually the best default)

In [ ]:
# Small dataset for clear dendrogram
X_hc, y_hc = make_blobs(n_samples=30, centers=3, cluster_std=0.6, random_state=5)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

linkage_methods = ['single', 'complete', 'average', 'ward']
titles = ['Single Linkage (Nearest)', 'Complete Linkage (Farthest)',
          'Average Linkage', "Ward's Linkage (min ΔWCSS)"]

for ax, method, title in zip(axes.flat, linkage_methods, titles):
    Z = linkage(X_hc, method=method)
    dendrogram(Z, ax=ax, leaf_rotation=90, leaf_font_size=8,
               color_threshold=0.7 * max(Z[:, 2]),
               above_threshold_color='gray')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Distance')
    ax.set_xlabel('Sample index')

plt.suptitle('Dendrograms with Different Linkage Criteria', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**What the plot tells us:** The four dendrograms reveal how linkage choice shapes the clustering hierarchy. **Single linkage** produces a "chaining" effect — it connects clusters through their nearest points, leading to long, straggly merges and unbalanced trees. **Complete linkage** produces more balanced clusters by using the farthest points, but can split natural clusters that are elongated. **Average linkage** is a compromise. **Ward's linkage** produces the most balanced, compact tree — the three main clusters are clearly separated by large jumps in height, making it easy to cut the dendrogram into 3 groups. Ward's is generally the best default for discovering compact clusters.

## 3.2 Cutting the Dendrogram

To obtain a flat clustering from a dendrogram, we **cut horizontally** at a chosen height. A higher cut produces fewer, coarser clusters; a lower cut produces more, finer ones.

In [ ]:
X_dend, y_dend = make_blobs(n_samples=50, centers=4, cluster_std=0.7, random_state=12)
Z_dend = linkage(X_dend, method='ward')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cut_heights = [8, 5, 2.5]
cut_labels = ['High cut → 2 clusters', 'Medium cut → 4 clusters', 'Low cut → 7 clusters']

for ax, h, label in zip(axes, cut_heights, cut_labels):
    dendrogram(Z_dend, ax=ax, leaf_rotation=90, leaf_font_size=7,
               color_threshold=h, above_threshold_color='gray')
    ax.axhline(y=h, color='#e63946', linestyle='--', linewidth=2.5, label=f'Cut at h={h}')
    n_clusters = len(np.unique(fcluster(Z_dend, t=h, criterion='distance')))
    ax.set_title(f'{label} ($K={n_clusters}$)', fontsize=12)
    ax.set_ylabel('Distance')
    ax.legend(fontsize=10)

plt.suptitle('Cutting the Dendrogram at Different Heights', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**What the plot tells us:** The same dendrogram is cut at three different heights. At height 8 (left), only the final two mega-merges are below the cut, yielding just 2 coarse clusters. At height 5 (middle), the cut captures the 4 natural groups — each coloured branch becomes a separate cluster. At height 2.5 (right), the cut is low enough to split some natural groups, producing 7 fine-grained clusters. The **choice of cut height is equivalent to choosing $K$**, but the dendrogram gives the analyst a full multi-scale picture to decide from, unlike K-Means where $K$ must be fixed upfront.

## 3.3 Hierarchical Clustering on Non-Convex Data

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.06, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
methods = ['single', 'complete', 'average', 'ward']

for ax, method in zip(axes, methods):
    agg = AgglomerativeClustering(n_clusters=2, linkage=method).fit(X_moons)
    ari = adjusted_rand_score(y_moons, agg.labels_)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=agg.labels_,
               cmap=ListedColormap(NTNU_COLORS[:2]), alpha=0.7, s=30, edgecolors='k', linewidth=0.2)
    ax.set_title(f'{method.capitalize()} (ARI={ari:.2f})', fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Hierarchical Clustering on Half-Moons Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**What the plot tells us:** **Single linkage achieves a perfect ARI of 1.00** on the half-moons — its "chaining" property, which was a weakness on compact data, is actually a strength here because it connects points through nearest-neighbour chains that follow the crescent shapes. Complete and Ward linkage both fail badly (low ARI) because they enforce compact, spherical clusters. Average linkage is intermediate. This is a great example of how **no single algorithm is best for all data geometries** — the right choice depends on the shape of the clusters you expect.

---
# 4. DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

## 4.1 Core Concepts

DBSCAN defines clusters as **maximal sets of density-connected points**. It requires two hyperparameters:
- $\varepsilon$: maximum radius of the neighbourhood
- $N$ (`min_samples`): minimum number of points in the ε-neighbourhood

Points are classified as:
- **Core point:** has $\geq N$ points within its $\varepsilon$-neighbourhood
- **Border point:** not core, but within $\varepsilon$ of a core point
- **Noise point:** neither core nor border — labelled as outlier

In [ ]:
# Demonstrate DBSCAN point types
np.random.seed(42)
X_db_demo = np.vstack([
    np.random.randn(80, 2) * 0.5 + [2, 2],
    np.random.randn(60, 2) * 0.5 + [5, 5],
    np.random.randn(5, 2) * 2 + [8, 1],   # noise
])

from sklearn.neighbors import NearestNeighbors
db = DBSCAN(eps=0.8, min_samples=5).fit(X_db_demo)
labels_db = db.labels_
core_mask = np.zeros(len(X_db_demo), dtype=bool)
core_mask[db.core_sample_indices_] = True
border_mask = ~core_mask & (labels_db != -1)
noise_mask = labels_db == -1

fig, ax = plt.subplots(figsize=(9, 7))

# Plot core points
ax.scatter(X_db_demo[core_mask, 0], X_db_demo[core_mask, 1],
           c=[NTNU_COLORS[l] for l in labels_db[core_mask]],
           s=50, alpha=0.8, edgecolors='k', linewidth=0.3, label='Core points', marker='o')

# Border points
ax.scatter(X_db_demo[border_mask, 0], X_db_demo[border_mask, 1],
           c=[NTNU_COLORS[l] for l in labels_db[border_mask]],
           s=80, alpha=0.8, edgecolors='k', linewidth=1, label='Border points', marker='s')

# Noise points
ax.scatter(X_db_demo[noise_mask, 0], X_db_demo[noise_mask, 1],
           c='gray', s=60, alpha=0.6, edgecolors='red', linewidth=1.5, label='Noise / Outliers', marker='X')

# Draw epsilon circles for a few core points
for idx in db.core_sample_indices_[::15]:
    circle = plt.Circle(X_db_demo[idx], 0.8, fill=False, color='gray', linestyle='--', alpha=0.3)
    ax.add_patch(circle)

ax.set_title(f'DBSCAN Point Types (ε=0.8, min_samples=5)\n{(labels_db != -1).sum()} clustered, {noise_mask.sum()} noise')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend(fontsize=11, loc='upper left')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**What the plot tells us:** DBSCAN has found 2 dense clusters (blue and red circles) and correctly identified the 5 scattered points in the lower-right as noise (gray ×-markers with red edges). The dashed circles show ε-neighbourhoods of selected core points — within each circle, there are at least 5 points, making them core points. The square markers are border points: they have fewer than 5 neighbours in their own ε-ball, but they lie within the ε-ball of at least one core point. **Unlike K-Means, DBSCAN does not force outliers into a cluster** — this explicit noise detection is one of its greatest strengths.

## 4.2 Choosing $\varepsilon$: The $k$-Distance Plot

A practical heuristic for selecting $\varepsilon$:
1. Compute the distance to each point's $k$-th nearest neighbour ($k$ = `min_samples`)
2. Sort in descending order and plot
3. Look for the "elbow" — the transition from sparse to dense

In [ ]:
X_kdist, _ = make_blobs(n_samples=500, centers=4, cluster_std=0.8, random_state=42)
# Add some noise
X_kdist = np.vstack([X_kdist, np.random.uniform(-8, 12, (30, 2))])

k = 10  # min_samples
nn = NearestNeighbors(n_neighbors=k).fit(X_kdist)
distances, _ = nn.kneighbors(X_kdist)
k_dists = np.sort(distances[:, -1])[::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# k-distance plot
axes[0].plot(range(len(k_dists)), k_dists, color=NTNU_BLUE, linewidth=2)
elbow_eps = 1.2  # visual estimate
axes[0].axhline(y=elbow_eps, color='#e63946', linestyle='--', linewidth=2, label=f'ε ≈ {elbow_eps}')
axes[0].set_xlabel('Points (sorted by $k$-distance)')
axes[0].set_ylabel(f'{k}-th nearest neighbour distance')
axes[0].set_title(f'$k$-Distance Plot ($k = {k}$)')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# DBSCAN result with that epsilon
db2 = DBSCAN(eps=elbow_eps, min_samples=k).fit(X_kdist)
labels_db2 = db2.labels_
n_clusters = len(set(labels_db2)) - (1 if -1 in labels_db2 else 0)
n_noise = (labels_db2 == -1).sum()

colors = [NTNU_COLORS[l] if l >= 0 else 'gray' for l in labels_db2]
axes[1].scatter(X_kdist[:, 0], X_kdist[:, 1], c=colors, s=30, alpha=0.7,
                edgecolors='k', linewidth=0.2)
noise_pts = X_kdist[labels_db2 == -1]
axes[1].scatter(noise_pts[:, 0], noise_pts[:, 1], c='gray', s=50, marker='X',
                edgecolors='red', linewidth=1, zorder=5, label=f'Noise ({n_noise} pts)')
axes[1].set_title(f'DBSCAN Result: {n_clusters} clusters, {n_noise} noise points')
axes[1].set_xlabel('$x_1$'); axes[1].set_ylabel('$x_2$')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**What the plot tells us:** The $k$-distance plot (left) shows a clear elbow around $\varepsilon \approx 1.2$. The steep portion at the left represents the few far-flung noise points (they have large distances to their 10th nearest neighbour). The flat portion at the right represents the dense cluster interiors. The elbow is the transition point. Using this $\varepsilon$ (right panel), DBSCAN correctly identifies 4 clusters and isolates the scattered uniform noise points as outliers. **The $k$-distance plot is the DBSCAN equivalent of the elbow method for K-Means.**

## 4.3 DBSCAN on Non-Convex Shapes

In [ ]:
X_moons2, y_moons2 = make_moons(n_samples=500, noise=0.08, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
eps_vals = [0.1, 0.2, 0.5]

for ax, eps in zip(axes, eps_vals):
    db_m = DBSCAN(eps=eps, min_samples=5).fit(X_moons2)
    labs = db_m.labels_
    n_cl = len(set(labs)) - (1 if -1 in labs else 0)
    n_ns = (labs == -1).sum()
    ari = adjusted_rand_score(y_moons2, labs) if n_cl > 1 else 0.0
    
    colors = [NTNU_COLORS[l % len(NTNU_COLORS)] if l >= 0 else 'lightgray' for l in labs]
    ax.scatter(X_moons2[:, 0], X_moons2[:, 1], c=colors, s=20, alpha=0.8,
               edgecolors='k', linewidth=0.15)
    ax.set_title(f'ε={eps}, {n_cl} clusters, {n_ns} noise\nARI={ari:.2f}', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('DBSCAN on Half-Moons: Effect of ε', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**What the plot tells us:** The parameter $\varepsilon$ critically determines DBSCAN's behaviour. At $\varepsilon=0.1$ (too small), most points cannot reach enough neighbours and the data is fragmented into many tiny clusters with substantial noise. At $\varepsilon=0.2$ (just right), the algorithm perfectly traces the two crescent shapes — achieving near-perfect ARI. At $\varepsilon=0.5$ (too large), the two moons become density-connected and merge into a single cluster. **DBSCAN excels at non-convex shapes but requires careful tuning of $\varepsilon$.**

---
# 5. Gaussian Mixture Models (GMMs)

## 5.1 From Hard to Soft Assignments

K-Means makes a **hard** assignment (each point belongs to exactly one cluster). GMMs provide a **probabilistic** alternative: each cluster is modelled as a Gaussian $\mathcal{N}(\boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$ and each point has a **soft membership** (probability of belonging to each component).

$$p(\boldsymbol{x}) = \sum_{k=1}^{K} \pi_k \; \mathcal{N}(\boldsymbol{x} \mid \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)$$

Parameters are learned via the **Expectation–Maximisation (EM)** algorithm.

In [ ]:
# Create anisotropic clusters to showcase GMM strengths
np.random.seed(42)
n_per = 150
# Cluster 1: elongated horizontal
c1 = np.random.randn(n_per, 2) @ [[2.0, 0.5], [0.0, 0.3]] + [0, 0]
# Cluster 2: elongated diagonal
c2 = np.random.randn(n_per, 2) @ [[0.5, 0.0], [1.0, 1.5]] + [5, 3]
# Cluster 3: small round
c3 = np.random.randn(n_per, 2) * 0.4 + [2, 5]

X_gmm = np.vstack([c1, c2, c3])
y_gmm = np.array([0]*n_per + [1]*n_per + [2]*n_per)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# K-Means
km_gmm = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_gmm)
axes[0].scatter(X_gmm[:, 0], X_gmm[:, 1], c=km_gmm.labels_,
                cmap=ListedColormap(NTNU_COLORS[:3]), alpha=0.6, s=20, edgecolors='k', linewidth=0.15)
axes[0].set_title(f'K-Means (ARI={adjusted_rand_score(y_gmm, km_gmm.labels_):.2f})')

# GMM — hard assignments
gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42).fit(X_gmm)
gmm_labels = gmm.predict(X_gmm)
axes[1].scatter(X_gmm[:, 0], X_gmm[:, 1], c=gmm_labels,
                cmap=ListedColormap(NTNU_COLORS[:3]), alpha=0.6, s=20, edgecolors='k', linewidth=0.15)
# Draw covariance ellipses
from matplotlib.patches import Ellipse
for k in range(3):
    mean = gmm.means_[k]
    cov = gmm.covariances_[k]
    v, w = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(w[1, 0], w[0, 0]))
    for n_std in [1, 2]:
        ell = Ellipse(xy=mean, width=2*n_std*np.sqrt(v[0]), height=2*n_std*np.sqrt(v[1]),
                      angle=angle, facecolor='none', edgecolor=NTNU_COLORS[k],
                      linewidth=2 if n_std==1 else 1, linestyle='-' if n_std==1 else '--')
        axes[1].add_patch(ell)
axes[1].set_title(f'GMM Hard Labels (ARI={adjusted_rand_score(y_gmm, gmm_labels):.2f})')

# GMM — soft assignments (probability heatmap)
probs = gmm.predict_proba(X_gmm)
max_prob = probs.max(axis=1)
sc = axes[2].scatter(X_gmm[:, 0], X_gmm[:, 1], c=max_prob,
                      cmap='RdYlGn', alpha=0.7, s=20, edgecolors='k', linewidth=0.15,
                      vmin=0.5, vmax=1.0)
plt.colorbar(sc, ax=axes[2], label='Max responsibility $\\gamma_{ik}$')
axes[2].set_title('GMM Soft Assignments (Confidence)')

for ax in axes:
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    ax.grid(True, alpha=0.2)

plt.suptitle('K-Means vs. GMM on Anisotropic Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**What the plot tells us:** K-Means (left) struggles because its spherical Voronoi cells cannot capture the elongated shapes — the horizontal cluster gets partly assigned to the wrong group, yielding a lower ARI. The GMM (middle) fits ellipsoidal covariance matrices to each component (shown as 1σ and 2σ ellipses), beautifully capturing the different orientations and sizes. Its ARI is notably higher. The right panel shows each point's **assignment confidence** — points deep inside a cluster are green (high confidence), while those in overlap regions are yellow/red (lower confidence). This **soft assignment** is a key advantage: it tells you which points are ambiguous, something K-Means cannot do.

## 5.2 Choosing $K$ with BIC

The **Bayesian Information Criterion** balances model fit against complexity:

$$\text{BIC} = -2\ln\hat{L} + p\ln m$$

In `scikit-learn`, a higher (less negative) BIC is better. We choose $K$ that maximises BIC.

In [ ]:
X_bic, y_bic = make_blobs(n_samples=500, centers=4, cluster_std=[0.8, 1.0, 0.6, 1.2], random_state=42)

K_range_bic = range(1, 10)
bic_scores = []
aic_scores = []

for k in K_range_bic:
    gmm_k = GaussianMixture(n_components=k, covariance_type='full', n_init=3, random_state=42).fit(X_bic)
    bic_scores.append(gmm_k.bic(X_bic))
    aic_scores.append(gmm_k.aic(X_bic))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(K_range_bic, bic_scores, 'o-', color=NTNU_BLUE, linewidth=2, markersize=8, label='BIC')
ax.plot(K_range_bic, aic_scores, 's--', color='#e63946', linewidth=2, markersize=7, label='AIC')

best_k_bic = list(K_range_bic)[np.argmin(bic_scores)]
ax.axvline(best_k_bic, color='#2a9d8f', linestyle=':', linewidth=2, label=f'Best $K$ (BIC) = {best_k_bic}')

ax.set_xlabel('Number of Components $K$')
ax.set_ylabel('Information Criterion (lower is better)')
ax.set_title('GMM Model Selection: BIC and AIC')
ax.set_xticks(list(K_range_bic))
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**What the plot tells us:** Both BIC (blue) and AIC (red) drop sharply from $K=1$ to $K=4$, indicating that the data is vastly better explained by 4 components than fewer. At $K=4$, BIC reaches its minimum — the best trade-off between model fit and complexity. For $K>4$, the BIC rises because the extra parameters (means, covariances, weights) are penalised more than the marginal improvement in likelihood. AIC is less conservative and stays flatter. **BIC correctly identifies $K=4$**, matching the true number of clusters we generated. BIC is generally preferred for model selection because its penalty scales with $\ln m$ and thus is better at preventing overfitting on larger datasets.

---
# 6. Method Comparison

Let us now compare **all four algorithms** side-by-side on several challenging datasets to see their strengths and weaknesses in action.

| | K-Means | Hierarchical | DBSCAN | GMM |
|---|---|---|---|---|
| **Cluster shape** | Spherical | Depends on linkage | Arbitrary | Ellipsoidal |
| **Needs K?** | Yes | No (cut later) | No | Yes (use BIC) |
| **Noise handling** | Poor | Limited | Strong | Moderate |
| **Assignments** | Hard | Hard | Hard + noise | Soft (probabilistic) |

In [ ]:
from sklearn.cluster import DBSCAN, AgglomerativeClustering

# Create challenging datasets
datasets = {
    'Blobs (easy)': make_blobs(n_samples=500, centers=3, cluster_std=0.8, random_state=42),
    'Half-Moons': make_moons(n_samples=500, noise=0.07, random_state=42),
    'Circles': make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=42),
}
# Varied density
np.random.seed(42)
X_var = np.vstack([
    np.random.randn(300, 2) * 0.3 + [0, 0],
    np.random.randn(100, 2) * 1.5 + [5, 5],
    np.random.uniform(-3, 8, (20, 2))  # noise
])
y_var = np.array([0]*300 + [1]*100 + [-1]*20)
datasets['Varied Density + Noise'] = (X_var, y_var)

algorithms = {
    'K-Means': lambda X, k: KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X),
    'Hierarchical\n(Ward)': lambda X, k: AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X),
    'DBSCAN': lambda X, k: DBSCAN(eps=0.3, min_samples=8).fit_predict(X),
    'GMM': lambda X, k: GaussianMixture(n_components=k, random_state=42).fit_predict(X),
}

fig, axes = plt.subplots(len(datasets), len(algorithms), figsize=(18, 16))

for row, (ds_name, (X_ds, y_ds)) in enumerate(datasets.items()):
    # Standardise for fair comparison
    X_scaled = StandardScaler().fit_transform(X_ds)
    n_true = len(set(y_ds)) - (1 if -1 in y_ds else 0)
    
    for col, (alg_name, alg_func) in enumerate(algorithms.items()):
        ax = axes[row, col]
        try:
            if 'DBSCAN' in alg_name:
                # Tune eps per dataset
                eps_map = {'Blobs (easy)': 0.35, 'Half-Moons': 0.2, 'Circles': 0.2, 'Varied Density + Noise': 0.5}
                labels_pred = DBSCAN(eps=eps_map.get(ds_name, 0.3), min_samples=8).fit_predict(X_scaled)
            else:
                labels_pred = alg_func(X_scaled, n_true)
            
            y_eval = y_ds.copy()
            # For ARI, exclude noise in ground truth if present
            mask_eval = y_eval >= 0
            if mask_eval.sum() > 0 and len(set(labels_pred[mask_eval])) > 1:
                ari = adjusted_rand_score(y_eval[mask_eval], labels_pred[mask_eval])
            else:
                ari = 0.0
        except:
            labels_pred = np.zeros(len(X_ds))
            ari = 0.0
        
        # Color mapping
        unique_labs = sorted(set(labels_pred))
        cmap_list = ['lightgray'] + NTNU_COLORS  # -1 gets gray
        colors_plot = []
        for l in labels_pred:
            if l == -1:
                colors_plot.append('lightgray')
            else:
                colors_plot.append(NTNU_COLORS[l % len(NTNU_COLORS)])
        
        ax.scatter(X_scaled[:, 0], X_scaled[:, 1], c=colors_plot, s=12, alpha=0.7,
                   edgecolors='k', linewidth=0.1)
        ax.set_title(f'ARI = {ari:.2f}', fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        
        if row == 0:
            ax.set_xlabel(alg_name, fontsize=12, fontweight='bold')
            ax.xaxis.set_label_position('top')
        if col == 0:
            ax.set_ylabel(ds_name, fontsize=11, fontweight='bold')

plt.suptitle('Algorithm Comparison Across Datasets', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**What the plot tells us:** This grid reveals each algorithm's character:

- **Blobs (row 1):** All algorithms perform well on this "easy" dataset — spherical, well-separated clusters are every algorithm's comfort zone.
- **Half-Moons (row 2):** K-Means and Ward's hierarchical fail because they assume convex shapes. DBSCAN and (sometimes) GMM handle the crescents better, with DBSCAN achieving the best result by tracing the density structure.
- **Circles (row 3):** Concentric rings are impossible for K-Means and GMM (which see a single blob). DBSCAN excels here by following the density contours.
- **Varied Density + Noise (row 4):** DBSCAN's explicit noise detection (gray points) is a clear advantage. K-Means and GMM are forced to assign every noisy point to a cluster, distorting the results.

**Key takeaway:** Start with K-Means as a baseline. If non-convex shapes are suspected, try DBSCAN. For uncertainty estimates or ellipsoidal clusters, use GMMs. Use hierarchical clustering for exploratory analysis.

---
# 7. Evaluation and Pitfalls

## 7.1 Internal vs. External Evaluation

- **Internal metrics** (no labels needed): Silhouette score, Davies–Bouldin index, WCSS
- **External metrics** (labels available): Adjusted Rand Index (ARI), Normalised Mutual Information (NMI)
- **Application value**: Are the clusters useful and interpretable?

In [ ]:
X_eval, y_eval = make_blobs(n_samples=400, centers=4, cluster_std=[0.5, 0.8, 1.0, 0.6], random_state=42)

methods_eval = {
    'K-Means (K=4)': KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(X_eval),
    'K-Means (K=2)': KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_eval),
    'K-Means (K=8)': KMeans(n_clusters=8, n_init=10, random_state=42).fit_predict(X_eval),
    'DBSCAN (ε=0.8)': DBSCAN(eps=0.8, min_samples=5).fit_predict(X_eval),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, labs) in zip(axes, methods_eval.items()):
    # Exclude noise for silhouette
    mask_valid = labs >= 0
    n_clusters_found = len(set(labs[mask_valid]))
    
    if n_clusters_found > 1:
        sil = silhouette_score(X_eval[mask_valid], labs[mask_valid])
        ari = adjusted_rand_score(y_eval[mask_valid], labs[mask_valid])
        nmi = normalized_mutual_info_score(y_eval[mask_valid], labs[mask_valid])
    else:
        sil, ari, nmi = 0, 0, 0
    
    colors = [NTNU_COLORS[l % len(NTNU_COLORS)] if l >= 0 else 'lightgray' for l in labs]
    ax.scatter(X_eval[:, 0], X_eval[:, 1], c=colors, s=20, alpha=0.7, edgecolors='k', linewidth=0.15)
    ax.set_title(f'{name}\nSil={sil:.2f} | ARI={ari:.2f} | NMI={nmi:.2f}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Evaluation Metrics Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**What the plot tells us:** K-Means with $K=4$ (the correct number) achieves the highest scores across all three metrics — high silhouette (compact and well-separated clusters), ARI ≈ 1 (near-perfect match with ground truth), and NMI ≈ 1. With $K=2$, the algorithm is forced to merge clusters, reducing all scores significantly. With $K=8$, over-splitting produces many small clusters that don't match the true structure — ARI and NMI drop while silhouette may still look reasonable (smaller clusters can be internally compact). DBSCAN produces competitive results with the right $\varepsilon$, plus it explicitly separates noise. **Using multiple metrics gives a more reliable picture than any single score.**

## 7.2 Common Pitfalls: Clustering Uniform Data

One of the most dangerous pitfalls is **forcing clusters where no meaningful groups exist**. K-Means will always produce $K$ clusters, even on purely random data.

In [ ]:
np.random.seed(42)
X_uniform = np.random.rand(300, 2) * 10

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, k in zip(axes, [1, 2, 4, 8]):
    if k == 1:
        ax.scatter(X_uniform[:, 0], X_uniform[:, 1], c='gray', alpha=0.6, s=25, edgecolors='k', linewidth=0.2)
        ax.set_title('Raw Data\n(no structure)', fontsize=11)
    else:
        km_u = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_uniform)
        sil = silhouette_score(X_uniform, km_u.labels_)
        ax.scatter(X_uniform[:, 0], X_uniform[:, 1], c=km_u.labels_,
                   cmap='Set2', alpha=0.6, s=25, edgecolors='k', linewidth=0.2)
        ax.scatter(km_u.cluster_centers_[:, 0], km_u.cluster_centers_[:, 1],
                   marker='X', s=150, c='black', zorder=5)
        ax.set_title(f'K-Means, K={k}\nSilhouette = {sil:.3f}', fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('⚠️ Pitfall: Forcing Clusters on Uniform (Structureless) Data', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**What the plot tells us:** The raw data (leftmost) is sampled from a uniform distribution — there is genuinely **no cluster structure**. Yet K-Means happily produces 2, 4, or 8 clusters by carving up the space into Voronoi regions. The silhouette scores are low and stagnant, reflecting the lack of real separation. **The algorithm does not tell you whether clusters are meaningful — it always finds $K$ groups.** This is why domain knowledge and multiple evaluation strategies are essential. Always ask: "Do these clusters make sense?", not just "What does the algorithm output?" 

## 7.3 Best Practices Summary

1. **Standardise** features when using distance-based methods
2. Think carefully about what **similarity should mean** for your data
3. Try **multiple algorithms** and compare
4. Test **sensitivity** to hyperparameters and initialisation
5. **Visualise** results (PCA/t-SNE projections, silhouette plots)
6. **Validate** against domain knowledge

> *Clustering is best treated as an exploratory process, not as automatic truth discovery.*

---

### Final Message

> Clustering does not discover *the one true structure* in data; it reveals structure *relative to the assumptions we choose*.

---
*End of Clustering Notebook — TTK4260, Spring 2026*